# Strands Agent

This notebook demonstrates how to build a ReAct-style agent using the **Strands Agents SDK** and **AWS Bedrock**. Strands takes a model-driven approach where the model decides what to do at each step. It is model-agnostic (Bedrock, Anthropic, OpenAI, Ollama) and has native MCP support, though this lab uses BedrockModel with locally defined tools.

You'll learn how to:
- Configure and connect to Claude models via Bedrock
- Define custom tools that the agent can use
- Build an agent that reasons and acts autonomously
- Test the agent with various queries

**Prerequisites:** Ensure your model is configured in `../CONFIG.txt` before running.

**Next:** Once you've tested the agent locally, proceed to `02_deploy_to_agentcore.ipynb` to deploy it as a managed service.

## 1. Setup

Check which packages are pre-installed, then install any missing dependencies. The key packages are:
- **strands-agents**: The Strands Agents SDK for building AI agents
- **strands-agents-tools**: Pre-built tools for common tasks

In [ ]:
import importlib.metadata

packages = [
    "strands-agents",
    "strands-agents-tools",
    "boto3",
]

print("Pre-installed packages:")
print("-" * 50)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:30} NOT INSTALLED")

In [ ]:
%pip install strands-agents strands-agents-tools -q

## 2. Imports

Import the required libraries:
- **strands**: `Agent` class and `tool` decorator for defining agent tools
- **strands.models**: `BedrockModel` for connecting to Claude on Amazon Bedrock

In [ ]:
from datetime import datetime

from strands import Agent, tool
from strands.models import BedrockModel

print("Imports successful!")

## 3. Configuration

Load the model ID and AWS region from `CONFIG.txt`. For cross-region inference profiles (MODEL_ID starting with `us.`), Bedrock routes requests to the optimal region automatically.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
REGION = os.getenv("REGION", "us-east-1")

print(f"Model ID: {MODEL_ID}")
print(f"Region:   {REGION}")

## 4. Define Tools

Tools are functions that the agent can call to perform actions or retrieve information. We use the `@tool` decorator from Strands to register them.

The agent will automatically decide when to use these tools based on the user's question.

In [ ]:
@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


print(f"Defined tools: get_current_time, add_numbers")

## 5. Create the Agent

Create a Strands `Agent` with:
- **BedrockModel**: Connects to Claude via Amazon Bedrock with temperature 0 for deterministic responses
- **Tools**: The custom tools we defined above
- **System prompt**: Instructions for the agent's behavior

Strands takes a model-driven approach: the model decides what to do at each step rather than following a developer-defined workflow. Strands handles the ReAct loop (Reason → Act → Observe → Repeat) internally. The agent decides when to use tools and when to respond directly.

In [ ]:
bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
)

agent = Agent(
    model=bedrock_model,
    system_prompt="You are a helpful assistant. Use the available tools when needed to answer questions.",
    tools=[get_current_time, add_numbers],
)

print(f"Agent created with model: {MODEL_ID}")

## 6. Run the Agent

The `run_agent` function sends a question to the agent and displays the response. The agent will automatically reason about whether to use tools, call them if needed, and synthesize a final answer.

In [ ]:
def run_agent(question: str):
    """Run the agent with a question and display the response."""
    print(f"Question: {question}")
    print("-" * 50)
    response = agent(question)
    print(f"\nResponse:\n{response}")
    return response

In [ ]:
# Test: Get current time
result = run_agent("What is the current time?")

In [ ]:
# Test: Math calculation
result = run_agent("What is 42 + 17?")

In [ ]:
# Test: Multiple tools
result = run_agent("What time is it and what is 100 + 200?")

## 7. Query with Sample Financial Data

This section demonstrates how to provide context to the agent by loading sample SEC 10-K filing data and asking questions about it.

This pattern is useful for:
- **RAG (Retrieval-Augmented Generation)**: Injecting retrieved documents into prompts
- **In-context learning**: Providing examples or data for the agent to reference
- **Domain-specific Q&A**: Answering questions based on financial filing information

In [ ]:
from load_sample_data import load_financial_data, print_info

financial_text = load_financial_data()
print_info(financial_text)

In [ ]:
def ask_about_data(question: str, context: str):
    """Ask the agent a question with context."""
    prompt = f"""Based on this SEC 10-K filing information:

{context}

Question: {question}"""
    return run_agent(prompt)

# Sample question about companies and products
ask_about_data("What companies are mentioned and what are their key products?", financial_text)

In [ ]:
# Another sample question about risk factors
ask_about_data("What risk factors are mentioned and which companies face them?", financial_text)